## Step 1: Load the dataset

In [49]:
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


### Step 1.2 — Download the dataset

In [50]:
!pip install -q gdown

import gdown
import pandas as pd

file_id = "1HWczIICsMpaL8EJyu48ZvRFcXx3_pcnb"

url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "sentiment_dataset.csv", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1HWczIICsMpaL8EJyu48ZvRFcXx3_pcnb
To: /content/sentiment_dataset.csv
100%|██████████| 620k/620k [00:00<00:00, 6.63MB/s]


'sentiment_dataset.csv'

### Step 1.3 — Load and inspect the dataset

In [51]:
df = pd.read_csv("sentiment_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (5937, 2)


,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


### Step 1.4 — Check the column names

In [52]:
print(df.columns)

Index(['Comment', 'Emotion'], dtype='object')


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB


## Step 2.1 — Understand the emotion labels

In [54]:
print(df['Emotion'].value_counts())

Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64


### Step 2.2 — Text Cleaning

In [55]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Clean_Comment'] = df['Comment'].apply(clean_text)

df[['Comment', 'Clean_Comment']].head()

,Comment,Clean_Comment
0,i seriously hate one subject to death but now ...,i seriously hate one subject to death but now ...
1,im so full of life i feel appalled,im so full of life i feel appalled
2,i sit here to write i start to dig out my feel...,i sit here to write i start to dig out my feel...
3,ive been really angry with r and i feel like a...,ive been really angry with r and i feel like a...
4,i feel suspicious if there is no one outside l...,i feel suspicious if there is no one outside l...


### Step 2.3 — Tokenization and Stopword Removal

In [56]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['Clean_Comment'] = df['Clean_Comment'].apply(preprocess_text)

df[['Comment', 'Clean_Comment']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Comment,Clean_Comment
0,i seriously hate one subject to death but now ...,seriously hate one subject death feel reluctan...
1,im so full of life i feel appalled,im full life feel appalled
2,i sit here to write i start to dig out my feel...,sit write start dig feelings think afraid acce...
3,ive been really angry with r and i feel like a...,ive really angry r feel like idiot trusting fi...
4,i feel suspicious if there is no one outside l...,feel suspicious one outside like rapture happe...


### Step 3- Feature Extraction and Train-Test Split

In [57]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Separate comments and emotion labels
X = df['Clean_Comment']
y = df['Emotion']

# Split the dataset
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create TF-IDF features using training data only
tfidf = TfidfVectorizer()

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("TF-IDF fitted on training data only.")

Training data: (4749, 7734)
Testing data: (1188, 7734)
TF-IDF fitted on training data only.


## Step 4 — Model Development

In [58]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Train SVM
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)

print("Both models trained successfully!")

Both models trained successfully!


### Step 5.1 — Evaluate the models
We'll use two important metrics:
- Accuracy: The percentage of comments classified correctly.
- F1-score: A measure that balances precision and recall.

In [59]:
from sklearn.metrics import accuracy_score, f1_score

# Predictions from both models
nb_predictions = nb_model.predict(X_test)
svm_predictions = svm_model.predict(X_test)

# Calculate evaluation metrics
print("Naive Bayes Results")
print("Accuracy:", accuracy_score(y_test, nb_predictions))
print("F1-score:", f1_score(y_test, nb_predictions, average='weighted'))

print("\nSVM Results")
print("Accuracy:", accuracy_score(y_test, svm_predictions))
print("F1-score:", f1_score(y_test, svm_predictions, average='weighted'))

Naive Bayes Results
Accuracy: 0.8897306397306397
F1-score: 0.889717452947724

SVM Results
Accuracy: 0.9410774410774411
F1-score: 0.9410487309807695


### Step 5.2 — Compare the Model Results

The Naive Bayes and Support Vector Machine (SVM) models were evaluated using accuracy, precision, recall, and F1-score.

| Evaluation Metric | Naive Bayes | SVM |
|---|---:|---:|
| Accuracy | 89% | 94% |
| Macro F1-score | 89% | 94% |
| Weighted F1-score | 89% | 94% |

**Observation:**

Naive Bayes achieved approximately 89% accuracy, while SVM achieved approximately 94% accuracy. SVM also performed well across all three emotion classes: anger, fear, and joy.

**Best-performing model:**

SVM performed better than Naive Bayes on the test dataset. It achieved approximately 5.13 percentage points higher accuracy, making it the preferred model for this emotion classification task.




### Step 5.3 — Generate the classification report

In [60]:
from sklearn.metrics import classification_report

print("Naive Bayes Classification Report")
print(classification_report(y_test, nb_predictions))

print("SVM Classification Report")
print(classification_report(y_test, svm_predictions))

Naive Bayes Classification Report
              precision    recall  f1-score   support

       anger       0.88      0.91      0.89       400
        fear       0.87      0.90      0.89       388
         joy       0.92      0.86      0.89       400

    accuracy                           0.89      1188
   macro avg       0.89      0.89      0.89      1188
weighted avg       0.89      0.89      0.89      1188

SVM Classification Report
              precision    recall  f1-score   support

       anger       0.95      0.93      0.94       400
        fear       0.94      0.93      0.94       388
         joy       0.93      0.96      0.95       400

    accuracy                           0.94      1188
   macro avg       0.94      0.94      0.94      1188
weighted avg       0.94      0.94      0.94      1188



## Step 6 — Conclusion

In this assignment, I developed machine learning models to classify emotions in text comments. The dataset contained three emotions: anger, joy, and fear.

I cleaned the text, performed tokenization and stopword removal, and used TF-IDF to convert text into numerical features. The TF-IDF vectorizer was fitted only on the training data.

I trained two models: Naive Bayes and Support Vector Machine (SVM). Naive Bayes achieved approximately 89% accuracy, while SVM achieved approximately 94% accuracy.

Based on the evaluation results, SVM performed better than Naive Bayes. Therefore, SVM was selected as the preferred model for this emotion classification task.
